# Expression-matched gene-set analysis: MT, Autophagy, and ncMT

This notebook extends `Mitochondrial_Expression_Matching.ipynb` to the two additional gene sets used in `cell_type_analysis.ipynb`:

- **Mitochondrial genes (MT):** defined exactly as in the original expression-matching notebook (`gene.lower().startswith("mt-")`).
- **Autophagy genes:** exact human/mouse/zebrafish gene-name variants from the cell-type notebook.
- **Nuclear mitochondrial genes (ncMT):** the exact `mitochondrial_related_genes` set labeled **Nuclear MT genes** in the cell-type notebook.

For each dataset and each gene set, random control sets are matched to the observed set on:

1. mean expression,
2. detection frequency,
3. expression variance.

The final matched sets have the **same number of genes as the observed set** and must be within **3% relative error for every matching variable**. The target is **1,000 unique matched sets per dataset and gene set**.

A fixed random seed is used for reproducibility. The old notebooks and output directory are left untouched.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.optimize import Bounds, LinearConstraint, milp
from statsmodels.stats.multitest import multipletests


# ============================================================
# DATA
# ============================================================

base_dir = Path(
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/"
    "preprocessing/pkl_files_filtered"
)

file_dict = {
    "treutlein": base_dir / "treutlein_all_genes.pkl",
    "weinreb": base_dir / "weinreb_all_genes.pkl",
    "junker": base_dir / "junker_z5_all_genes.pkl",

    "you1": base_dir / "you_stage1_barcode_all_genes.pkl",
    "you2": base_dir / "you_stage2_barcode_all_genes.pkl",
    "you3": base_dir / "you_stage3_barcode_all_genes.pkl",
    "you4": base_dir / "you_stage4_barcode_all_genes.pkl",

    "knoblich_day16_1": base_dir / "knoblich_day16_1.pkl",
    "knoblich_day16_2": base_dir / "knoblich_day16_2.pkl",
    "knoblich_day42_1": base_dir / "knoblich_day42_1.pkl",
    "knoblich_day42_2": base_dir / "knoblich_day42_2.pkl",
}

name_map = {
    "treutlein": "HBO1",
    "weinreb": "MHSPC",
    "junker": "ZFL",
    "knoblich_day16_1": "HBO2_16_1",
    "knoblich_day16_2": "HBO2_16_2",
    "knoblich_day42_1": "HBO2_42_1",
    "knoblich_day42_2": "HBO2_42_2",
    "you1": "HBO3_1",
    "you2": "HBO3_2",
    "you3": "HBO3_3",
    "you4": "HBO3_4",
}

dataset_order = [
    "HBO1", "MHSPC", "ZFL",
    "HBO2_16_1", "HBO2_16_2", "HBO2_42_1", "HBO2_42_2",
    "HBO3_1", "HBO3_2", "HBO3_3", "HBO3_4",
]


# ============================================================
# NEW OUTPUT DIRECTORY — does not overwrite MT-only analysis
# ============================================================

MATCH_OUT_DIR = Path(
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/analysis/"
    "MT_autophagy_ncMT_expression_matching"
)
MATCH_OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# MATCHING SETTINGS
# ============================================================

N_MATCHED_SETS = 1000
MATCH_TOLERANCE = 0.03
SEED_TOLERANCE = 0.01
RANDOM_SEED = 42
MAX_RANDOM_WALK_STEPS = 200000

# Mirrors the old MT notebook: when matching one target set,
# only genes from that target set are excluded from controls.
# Set True if you instead want MT/autophagy/ncMT all excluded
# from every control pool.
EXCLUDE_ALL_SPECIAL_GENE_SETS_FROM_CONTROLS = False

GENE_SET_ORDER = ["MT", "Autophagy", "ncMT"]

# Deterministic offsets so every dataset/gene-set combination gets
# its own reproducible random stream.
GENE_SET_SEED_OFFSET = {
    "MT": 0,
    "Autophagy": 100000,
    "ncMT": 200000,
}


## Gene-set definitions

In [2]:
# ============================================================
# AUTOPHAGY GENES
# Exact definitions from cell_type_analysis.ipynb
# ============================================================

autophagy_genes = [
    "ATG5", "ATG7", "BECN1", "MAP1LC3B", "SQSTM1", "GABARAPL1",
    "ULK1", "WIPI2", "ATG12", "ATG16L1", "VPS34", "UVRAG",
    "PINK1", "PRKN", "BNIP3", "DRAM1", "GABARAP", "LC3B",
    "ATG4B", "RUBCN", "FIP200", "AMBRA1"
]

zebrafish_autophagy_genes = [
    "atg5", "atg7", "becn1", "map1lc3b", "sqstm1", "gabarapl1",
    "ulk1", "wipi2", "atg12", "atg16l1", "pik3c3", "uvrag",
    "pink1", "parkin", "bnip3", "dram1", "gabarap", "map1lc3b",
    "atg4b", "rubcn", "rb1cc1", "ambra1"
]

mouse_autophagy_genes = [
    "Atg5", "Atg7", "Becn1", "Map1lc3b", "Sqstm1", "Gabarapl1",
    "Ulk1", "Wipi2", "Atg12", "Atg16l1", "Pik3c3", "Uvrag",
    "Pink1", "Prkn", "Bnip3", "Dram1", "Gabarap", "Map1lc3b",
    "Atg4b", "Rubcn", "Rb1cc1", "Ambra1"
]

AUTOPHAGY_GENES = set(
    autophagy_genes
    + zebrafish_autophagy_genes
    + mouse_autophagy_genes
)


# ============================================================
# NUCLEAR-ENCODED MITOCHONDRIAL / ncMT GENES
# Exact mitochondrial_related_genes definitions from
# cell_type_analysis.ipynb (shown there as "Nuclear MT genes")
# ============================================================

zebrafish_ncmt_genes = [
    "ndufs1", "sdha", "uqcrc1", "cox4i1", "atp5f1a",
    "ppargc1a", "tfam", "twinkle", "polg",
    "tomm20a", "timm23", "hspa9",
    "mrpl12", "mrps22",
    "sod2", "gpx4a", "prdx3",
    "mfn1", "mfn2", "opa1", "dnm1l",
    "bcl2", "cycs",
    "idh3a", "cpt1a", "glud1a",
    "atf4a", "chop"
]

mouse_ncmt_genes = [
    "Ndufs1", "Sdha", "Uqcrc1", "Cox4i1", "Atp5f1a",
    "Ppargc1a", "Tfam", "Twnk", "Polg",
    "Tomm20", "Timm23", "Hspa9",
    "Mrpl12", "Mrps22",
    "Sod2", "Gpx4", "Prdx3",
    "Mfn1", "Mfn2", "Opa1", "Dnm1l",
    "Bcl2", "Cycs",
    "Idh3a", "Cpt1a", "Glud1",
    "Atf4", "Chop"
]

human_ncmt_genes = [
    "NDUFS1", "SDHA", "UQCRC1", "COX4I1", "ATP5F1A",
    "PPARGC1A", "TFAM", "TWNK", "POLG",
    "TOMM20", "TIMM23", "HSPA9",
    "MRPL12", "MRPS22",
    "SOD2", "GPX4", "PRDX3",
    "MFN1", "MFN2", "OPA1", "DNM1L",
    "BCL2", "CYCS",
    "IDH3A", "CPT1A", "GLUD1",
    "ATF4", "CHOP"
]

NCMT_GENES = set(
    human_ncmt_genes
    + zebrafish_ncmt_genes
    + mouse_ncmt_genes
)


## Calculate gene-level matching covariates

In [3]:
def calculate_gene_covariates(full_df):
    """
    Last dataframe column = lineage barcode.

    Returns one row per unique gene with:
        - mean expression
        - detection frequency
        - expression variance
        - membership in MT / Autophagy / ncMT

    MT membership is intentionally defined exactly as in the original
    Mitochondrial_Expression_Matching notebook.
    """

    gene_data = full_df.iloc[:, :-1]

    rows = []

    for j, gene in enumerate(gene_data.columns):
        x = gene_data.iloc[:, j].to_numpy()

        rows.append({
            "gene": str(gene),
            "mean_expression": np.mean(x),
            "detection_frequency": np.mean(x > 0),
            "expression_variance": np.var(x),
        })

    gene_stats = pd.DataFrame(rows)

    # Collapse duplicate gene names exactly as in the MT notebook.
    gene_stats = (
        gene_stats
        .groupby("gene", as_index=False)
        .agg({
            "mean_expression": "mean",
            "detection_frequency": "mean",
            "expression_variance": "mean",
        })
    )

    gene_stats["is_MT"] = (
        gene_stats["gene"]
        .str.lower()
        .str.startswith("mt-")
    )

    gene_stats["is_Autophagy"] = gene_stats["gene"].isin(AUTOPHAGY_GENES)
    gene_stats["is_ncMT"] = gene_stats["gene"].isin(NCMT_GENES)

    return gene_stats


def target_column(gene_set):
    return {
        "MT": "is_MT",
        "Autophagy": "is_Autophagy",
        "ncMT": "is_ncMT",
    }[gene_set]


def get_target_and_control_df(gene_stats, gene_set):
    """Return observed target genes and the eligible control pool."""

    target_col = target_column(gene_set)
    target_df = gene_stats.loc[gene_stats[target_col]].copy().reset_index(drop=True)

    if EXCLUDE_ALL_SPECIAL_GENE_SETS_FROM_CONTROLS:
        special_mask = gene_stats[["is_MT", "is_Autophagy", "is_ncMT"]].any(axis=1)
        ctrl_df = gene_stats.loc[~special_mask].copy().reset_index(drop=True)
    else:
        # This mirrors the original MT notebook: exclude only the target set.
        ctrl_df = gene_stats.loc[~gene_stats[target_col]].copy().reset_index(drop=True)

    if len(target_df) == 0:
        raise ValueError(f"No {gene_set} genes are present in this dataset.")

    if len(ctrl_df) < len(target_df):
        raise ValueError(
            f"Not enough control genes for {gene_set}: "
            f"{len(ctrl_df)} controls for {len(target_df)} target genes."
        )

    return target_df, ctrl_df


## Sanity check: genes present in each dataset

In [4]:
gene_count_rows = []

for dataset, pkl_path in file_dict.items():
    full_df = pd.read_pickle(pkl_path)
    gene_stats = calculate_gene_covariates(full_df)
    del full_df

    for gene_set in GENE_SET_ORDER:
        target_df, ctrl_df = get_target_and_control_df(gene_stats, gene_set)

        gene_count_rows.append({
            "dataset": dataset,
            "dataset_label": name_map[dataset],
            "gene_set": gene_set,
            "n_target_genes": len(target_df),
            "n_control_pool": len(ctrl_df),
            "target_genes": ", ".join(target_df["gene"].astype(str).tolist()),
        })

gene_count_df = pd.DataFrame(gene_count_rows)
display(gene_count_df[[
    "dataset_label", "gene_set", "n_target_genes", "n_control_pool"
]])


,dataset_label,gene_set,n_target_genes,n_control_pool
0,HBO1,MT,13,20843
1,HBO1,Autophagy,18,20838
2,HBO1,ncMT,25,20831
3,MHSPC,MT,13,22623
4,MHSPC,Autophagy,20,22616
5,MHSPC,ncMT,25,22611
6,ZFL,MT,13,21049
7,ZFL,Autophagy,15,21047
8,ZFL,ncMT,19,21043
9,HBO3_1,MT,13,25336


## Find a reproducible feasible seed with MILP


The MILP selects exactly the same number of control genes as the target set and constrains the three set-level means to fall within a tight seed tolerance. It first tries 1%; if that is infeasible it falls back to the final 3% tolerance.

In [5]:
MATCH_METRICS = [
    "mean_expression",
    "detection_frequency",
    "expression_variance",
]


def find_feasible_seed_milp(
    gene_stats,
    gene_set,
    seed_tolerance=SEED_TOLERANCE,
    fallback_tolerance=MATCH_TOLERANCE,
):
    """
    Find one feasible matched control set using scipy.optimize.milp.

    The optimization contains no MI information.
    """

    target_df, ctrl_df = get_target_and_control_df(gene_stats, gene_set)

    n_target = len(target_df)
    target = target_df[MATCH_METRICS].mean().to_numpy(dtype=float)
    scale = np.maximum(np.abs(target), 1e-12)

    ctrl_values = ctrl_df[MATCH_METRICS].to_numpy(dtype=float)

    # Small objective: prefer genes whose covariates are individually
    # reasonably close to the target. Feasibility constraints determine
    # the actual set-level match.
    standardized_distance = np.sum(
        ((ctrl_values - target[None, :]) / scale[None, :]) ** 2,
        axis=1,
    )

    n_ctrl = len(ctrl_df)
    integrality = np.ones(n_ctrl, dtype=int)
    bounds = Bounds(np.zeros(n_ctrl), np.ones(n_ctrl))

    # Rows: exact set size + one row per matching metric.
    A = np.vstack([
        np.ones(n_ctrl),
        ctrl_values.T,
    ])

    for tolerance in [seed_tolerance, fallback_tolerance]:
        lower_metric = n_target * target * (1 - tolerance)
        upper_metric = n_target * target * (1 + tolerance)

        lb = np.concatenate([[n_target], lower_metric])
        ub = np.concatenate([[n_target], upper_metric])

        constraints = LinearConstraint(A, lb, ub)

        result = milp(
            c=standardized_distance,
            integrality=integrality,
            bounds=bounds,
            constraints=constraints,
            options={"disp": False},
        )

        if result.success and result.x is not None:
            selected_idx = np.flatnonzero(result.x > 0.5)

            if len(selected_idx) != n_target:
                raise RuntimeError(
                    f"MILP returned {len(selected_idx)} genes; expected {n_target}."
                )

            seed_genes = ctrl_df.iloc[selected_idx]["gene"].astype(str).tolist()

            selected_means = ctrl_df.iloc[selected_idx][MATCH_METRICS].mean().to_numpy()
            signed_error = (selected_means - target) / scale
            max_error = np.abs(signed_error).max()

            print(
                f"{gene_set}: MILP seed found at {100*tolerance:.1f}% constraint | "
                f"actual max error = {100*max_error:.3f}%"
            )

            return seed_genes

    raise RuntimeError(
        f"Could not find a feasible MILP seed for {gene_set} even at "
        f"{100*fallback_tolerance:.1f}% tolerance."
    )


## Expand each seed to 1,000 feasible matched sets

In [6]:
def expand_feasible_matched_sets(
    gene_stats,
    gene_set,
    seed_genes,
    n_sets=N_MATCHED_SETS,
    tolerance=MATCH_TOLERANCE,
    random_seed=RANDOM_SEED,
    max_steps=MAX_RANDOM_WALK_STEPS,
):
    """
    Generate diverse matched control sets by random walking through
    feasible set space.

    Every retained set:
        - has exactly the same number of genes as the observed gene set
        - contains no genes from the current target set
        - differs by <= tolerance from the target set for:
            * mean expression
            * detection frequency
            * expression variance

    MI is never used in matching.
    """

    rng = np.random.default_rng(random_seed)

    target_df, ctrl_df = get_target_and_control_df(gene_stats, gene_set)
    n_target = len(target_df)

    target = target_df[MATCH_METRICS].mean().to_numpy(dtype=float)
    scale = np.maximum(np.abs(target), 1e-12)

    ctrl_values = ctrl_df[MATCH_METRICS].to_numpy(dtype=float)
    ctrl_genes = ctrl_df["gene"].astype(str).to_numpy()

    gene_to_idx = {gene: i for i, gene in enumerate(ctrl_genes)}

    missing = [g for g in seed_genes if g not in gene_to_idx]
    if missing:
        raise ValueError(f"Seed genes missing from control pool: {missing}")

    selected = np.array([gene_to_idx[g] for g in seed_genes], dtype=int)

    if len(selected) != n_target:
        raise ValueError(
            f"Seed contains {len(selected)} genes; expected {n_target}."
        )

    current_sum = ctrl_values[selected].sum(axis=0)

    def evaluate_sum(xsum):
        means = xsum / n_target
        signed_error = (means - target) / scale
        max_error = np.abs(signed_error).max()
        return max_error, means, signed_error

    score, _, _ = evaluate_sum(current_sum)
    if score > tolerance:
        raise ValueError(
            f"Starting seed is not feasible: {100*score:.3f}% maximum mismatch."
        )

    seen = set()
    set_rows = []
    gene_rows = []

    def save_current_set():
        key = tuple(sorted(ctrl_genes[selected]))

        if key in seen:
            return False

        score, means, errors = evaluate_sum(current_sum)
        if score > tolerance:
            return False

        seen.add(key)
        set_id = len(seen)

        set_rows.append({
            "set_id": set_id,
            "mean_expression": means[0],
            "detection_frequency": means[1],
            "expression_variance": means[2],
            "rel_error_expression": errors[0],
            "rel_error_detection": errors[1],
            "rel_error_variance": errors[2],
            "max_rel_error": score,
        })

        for gene in ctrl_genes[selected]:
            gene_rows.append({"set_id": set_id, "gene": gene})

        return True

    save_current_set()

    steps = 0

    while len(seen) < n_sets and steps < max_steps:
        steps += 1

        out_pos = rng.integers(0, n_target)
        old_idx = selected[out_pos]
        sum_without = current_sum - ctrl_values[old_idx]

        # Evaluate all possible one-gene replacements simultaneously.
        proposed_means = (sum_without[None, :] + ctrl_values) / n_target
        relative_errors = (proposed_means - target[None, :]) / scale[None, :]
        max_errors = np.max(np.abs(relative_errors), axis=1)

        valid = max_errors <= tolerance
        valid[selected] = False

        candidate_idx = np.flatnonzero(valid)
        if len(candidate_idx) == 0:
            continue

        # Prefer better-matched replacements while retaining diversity.
        candidate_scores = max_errors[candidate_idx]
        weights = tolerance - candidate_scores + 1e-6
        weights = weights / weights.sum()

        new_idx = rng.choice(candidate_idx, p=weights)
        selected[out_pos] = new_idx
        current_sum = sum_without + ctrl_values[new_idx]

        save_current_set()

        if steps % 1000 == 0:
            print(
                f"{gene_set} | steps: {steps} | "
                f"unique sets: {len(seen)}/{n_sets}",
                end="\r",
            )

    print()
    print(
        f"{gene_set}: finished with {len(seen)} unique sets "
        f"after {steps} feasible-swap steps."
    )

    if len(seen) < n_sets:
        raise RuntimeError(
            f"{gene_set}: only generated {len(seen)}/{n_sets} matched sets. "
            "Increase MAX_RANDOM_WALK_STEPS or inspect the feasible space."
        )

    return pd.DataFrame(set_rows), pd.DataFrame(gene_rows)


## Run matching for all 3 gene sets × 11 datasets

In [ ]:
all_matched_qc = []
all_observed_qc = []

for dataset_index, (dataset, pkl_path) in enumerate(file_dict.items()):
    print("\n" + "=" * 80)
    print(dataset, "|", name_map[dataset])
    print("=" * 80)

    full_df = pd.read_pickle(pkl_path)
    gene_stats = calculate_gene_covariates(full_df)
    del full_df

    for gene_set in GENE_SET_ORDER:
        print("\n" + "-" * 60)
        print(gene_set)
        print("-" * 60)

        target_df, _ = get_target_and_control_df(gene_stats, gene_set)

        print("Observed genes:", len(target_df))
        print(target_df["gene"].astype(str).tolist())

        # ----------------------------------------------------
        # Reproducible seed
        # ----------------------------------------------------
        seed_genes = find_feasible_seed_milp(
            gene_stats=gene_stats,
            gene_set=gene_set,
        )

        # ----------------------------------------------------
        # Reproducible feasible random walk
        # ----------------------------------------------------
        this_seed = (
            RANDOM_SEED
            + GENE_SET_SEED_OFFSET[gene_set]
            + dataset_index * 1000
        )

        set_summary, gene_selections = expand_feasible_matched_sets(
            gene_stats=gene_stats,
            gene_set=gene_set,
            seed_genes=seed_genes,
            n_sets=N_MATCHED_SETS,
            tolerance=MATCH_TOLERANCE,
            random_seed=this_seed,
            max_steps=MAX_RANDOM_WALK_STEPS,
        )

        # ----------------------------------------------------
        # Save separately by gene set
        # ----------------------------------------------------
        gene_set_dir = MATCH_OUT_DIR / gene_set
        gene_set_dir.mkdir(parents=True, exist_ok=True)

        set_summary.to_csv(
            gene_set_dir / f"{dataset}_matched_set_QC_FINAL.csv",
            index=False,
        )

        gene_selections.to_csv(
            gene_set_dir / f"{dataset}_matched_gene_sets_FINAL.csv",
            index=False,
        )

        # ----------------------------------------------------
        # Combined QC objects
        # ----------------------------------------------------
        set_summary["dataset"] = dataset
        set_summary["dataset_label"] = name_map[dataset]
        set_summary["gene_set"] = gene_set
        all_matched_qc.append(set_summary)

        all_observed_qc.append(pd.DataFrame({
            "dataset": [dataset],
            "dataset_label": [name_map[dataset]],
            "gene_set": [gene_set],
            "n_genes": [len(target_df)],
            "mean_expression": [target_df["mean_expression"].mean()],
            "detection_frequency": [target_df["detection_frequency"].mean()],
            "expression_variance": [target_df["expression_variance"].mean()],
        }))

matched_set_qc_final = pd.concat(all_matched_qc, ignore_index=True)
observed_qc_final = pd.concat(all_observed_qc, ignore_index=True)



treutlein | HBO1

------------------------------------------------------------
MT
------------------------------------------------------------
Observed genes: 13
['MT-ATP6', 'MT-ATP8', 'MT-CO1', 'MT-CO2', 'MT-CO3', 'MT-CYB', 'MT-ND1', 'MT-ND2', 'MT-ND3', 'MT-ND4', 'MT-ND4L', 'MT-ND5', 'MT-ND6']
MT: MILP seed found at 1.0% constraint | actual max error = 0.954%

MT: finished with 1000 unique sets after 1286 feasible-swap steps.

------------------------------------------------------------
Autophagy
------------------------------------------------------------
Observed genes: 18
['AMBRA1', 'ATG12', 'ATG16L1', 'ATG4B', 'ATG5', 'ATG7', 'BECN1', 'BNIP3', 'DRAM1', 'GABARAP', 'GABARAPL1', 'MAP1LC3B', 'PINK1', 'RUBCN', 'SQSTM1', 'ULK1', 'UVRAG', 'WIPI2']
Autophagy: MILP seed found at 1.0% constraint | actual max error = 1.000%
Autophagy | steps: 1000 | unique sets: 999/1000
Autophagy: finished with 1000 unique sets after 1001 feasible-swap steps.

----------------------------------------------

## Matching QC

In [ ]:
qc_summary = (
    matched_set_qc_final
    .groupby(["gene_set", "dataset_label"])
    .agg(
        n_sets=("set_id", "nunique"),
        mean_expr_error=("rel_error_expression", "mean"),
        max_expr_error=("rel_error_expression", lambda x: np.abs(x).max()),
        mean_detection_error=("rel_error_detection", "mean"),
        max_detection_error=("rel_error_detection", lambda x: np.abs(x).max()),
        mean_variance_error=("rel_error_variance", "mean"),
        max_variance_error=("rel_error_variance", lambda x: np.abs(x).max()),
        max_overall_error=("max_rel_error", "max"),
    )
    .reset_index()
)

error_cols = [c for c in qc_summary.columns if "error" in c]
qc_summary[error_cols] *= 100

display(qc_summary)

assert (qc_summary["n_sets"] == N_MATCHED_SETS).all()
assert (qc_summary["max_overall_error"] <= 100 * MATCH_TOLERANCE + 1e-6).all()

print(
    f"All dataset × gene-set combinations have {N_MATCHED_SETS} sets "
    f"and all sets satisfy the {100*MATCH_TOLERANCE:.1f}% criterion."
)


In [ ]:
def plot_matching_qc(gene_set, metric, ylabel, log_scale=False):
    plot_sets = matched_set_qc_final.loc[
        matched_set_qc_final["gene_set"] == gene_set
    ].copy()

    plot_obs = observed_qc_final.loc[
        observed_qc_final["gene_set"] == gene_set
    ].copy()

    fig, ax = plt.subplots(figsize=(12, 5))

    sns.violinplot(
        data=plot_sets,
        x="dataset_label",
        y=metric,
        order=dataset_order,
        inner=None,
        cut=0,
        color="lightgray",
        ax=ax,
    )

    for i, dataset_label in enumerate(dataset_order):
        value = plot_obs.loc[
            plot_obs["dataset_label"] == dataset_label,
            metric,
        ].iloc[0]

        ax.scatter(
            i,
            value,
            color="black",
            s=55,
            zorder=5,
            label=f"{gene_set} genes" if i == 0 else None,
        )

    ax.set_title(gene_set)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

    if log_scale:
        ax.set_yscale("log")

    plt.xticks(rotation=45, ha="right")
    ax.grid(False)
    sns.despine(ax=ax)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


for gene_set in GENE_SET_ORDER:
    plot_matching_qc(gene_set, "mean_expression", "Mean expression")
    plot_matching_qc(gene_set, "detection_frequency", "Detection frequency")
    plot_matching_qc(
        gene_set,
        "expression_variance",
        "Expression variance",
        log_scale=True,
    )


## Compare MI of observed vs expression-matched gene sets

This section mirrors the MT notebook and uses `true_mi`. For each observed gene set, the empirical one-sided test asks whether its mean MI is higher than the distribution of mean MI from the 1,000 expression/detection/variance-matched control sets.

In [ ]:
MI_COL = "true_mi"

MI_DIR = Path(
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/calc_files/MI/"
    "mi_analysis_results_MI_label_entropy_all_genes_parallel"
)

MI_MATCH_OUT_DIR = MATCH_OUT_DIR / "MI_results"
MI_MATCH_OUT_DIR.mkdir(parents=True, exist_ok=True)

all_summaries = []

for dataset, pkl_path in file_dict.items():
    print("\n" + "=" * 80)
    print(dataset, "|", name_map[dataset])
    print("=" * 80)

    mi_path = MI_DIR / f"MI_label_entropy_{dataset}_50_permutations.csv"
    mi_df = pd.read_csv(mi_path)

    if "gene" not in mi_df.columns or MI_COL not in mi_df.columns:
        raise ValueError(
            f"{dataset}: required columns gene/{MI_COL} missing. "
            f"Columns: {list(mi_df.columns)}"
        )

    mi_lookup = (
        mi_df[["gene", MI_COL]]
        .assign(gene=lambda x: x["gene"].astype(str))
        .groupby("gene", as_index=False)[MI_COL]
        .max()
        .set_index("gene")[MI_COL]
    )

    full_df = pd.read_pickle(pkl_path)
    gene_stats = calculate_gene_covariates(full_df)
    del full_df

    for gene_set in GENE_SET_ORDER:
        print("\n", gene_set)

        target_df, _ = get_target_and_control_df(gene_stats, gene_set)
        target_genes = target_df["gene"].astype(str).tolist()

        missing_target = [g for g in target_genes if g not in mi_lookup.index]
        if missing_target:
            raise ValueError(
                f"{dataset} / {gene_set}: target genes missing MI: {missing_target}"
            )

        target_mi_values = mi_lookup.loc[target_genes].to_numpy()
        observed_mean_mi = target_mi_values.mean()
        observed_median_mi = np.median(target_mi_values)

        matched_genes_path = (
            MATCH_OUT_DIR / gene_set / f"{dataset}_matched_gene_sets_FINAL.csv"
        )
        matched_genes = pd.read_csv(matched_genes_path)
        matched_genes["gene"] = matched_genes["gene"].astype(str)
        matched_genes["mi"] = matched_genes["gene"].map(mi_lookup)

        n_missing = matched_genes["mi"].isna().sum()
        if n_missing > 0:
            missing_genes = matched_genes.loc[
                matched_genes["mi"].isna(), "gene"
            ].unique()
            raise ValueError(
                f"{dataset} / {gene_set}: matched genes missing MI: {missing_genes}"
            )

        matched_set_mi = (
            matched_genes
            .groupby("set_id", as_index=False)
            .agg(
                mean_mi=("mi", "mean"),
                median_mi=("mi", "median"),
                n_genes=("gene", "nunique"),
            )
        )

        if len(matched_set_mi) != N_MATCHED_SETS:
            raise ValueError(
                f"{dataset} / {gene_set}: found {len(matched_set_mi)} MI sets, "
                f"expected {N_MATCHED_SETS}."
            )

        if not (matched_set_mi["n_genes"] == len(target_genes)).all():
            raise ValueError(
                f"{dataset} / {gene_set}: matched-set gene counts do not equal "
                f"observed set size {len(target_genes)}."
            )

        n_sets = len(matched_set_mi)

        empirical_p_mean = (
            1 + np.sum(matched_set_mi["mean_mi"] >= observed_mean_mi)
        ) / (n_sets + 1)

        empirical_p_median = (
            1 + np.sum(matched_set_mi["median_mi"] >= observed_median_mi)
        ) / (n_sets + 1)

        matched_mean_mi = matched_set_mi["mean_mi"].mean()
        matched_median_mi = matched_set_mi["median_mi"].mean()

        mi_ratio = observed_mean_mi / matched_mean_mi
        mi_difference = observed_mean_mi - matched_mean_mi

        matched_set_mi.to_csv(
            MI_MATCH_OUT_DIR /
            f"{dataset}_{gene_set}_expression_matched_MI_distribution.csv",
            index=False,
        )

        all_summaries.append({
            "dataset": dataset,
            "dataset_label": name_map[dataset],
            "gene_set": gene_set,
            "n_target_genes": len(target_genes),
            "n_matched_sets": n_sets,
            "observed_mean_MI": observed_mean_mi,
            "matched_mean_MI": matched_mean_mi,
            "MI_difference": mi_difference,
            "MI_ratio_observed_over_matched": mi_ratio,
            "empirical_p_mean": empirical_p_mean,
            "observed_median_MI": observed_median_mi,
            "matched_median_MI": matched_median_mi,
            "empirical_p_median": empirical_p_median,
        })

        print("Observed mean MI:", observed_mean_mi)
        print("Matched mean MI:", matched_mean_mi)
        print("Ratio:", mi_ratio)
        print("Empirical one-sided p:", empirical_p_mean)


## Multiple-testing correction

Two BH columns are saved deliberately:

- `p_BH_within_gene_set`: 11 tests at a time. This is the direct analogue of the MT-only notebook, which corrected across the 11 datasets for the MT hypothesis.
- `p_BH_all_33`: all 3 gene sets × 11 datasets together (33 tests), if you regard all three gene-set analyses as one hypothesis family.

This avoids silently choosing one family definition for you.

In [ ]:
mi_summary_df = pd.DataFrame(all_summaries)

# ============================================================
# BH within each gene set: 11 dataset-level tests at a time
# ============================================================

mi_summary_df["p_BH_within_gene_set"] = np.nan

for gene_set, idx in mi_summary_df.groupby("gene_set").groups.items():
    mi_summary_df.loc[idx, "p_BH_within_gene_set"] = multipletests(
        mi_summary_df.loc[idx, "empirical_p_mean"],
        method="fdr_bh",
    )[1]

# ============================================================
# BH across all 33 gene-set × dataset tests
# ============================================================

mi_summary_df["p_BH_all_33"] = multipletests(
    mi_summary_df["empirical_p_mean"],
    method="fdr_bh",
)[1]

mi_summary_df["significant_BH_within_gene_set"] = (
    mi_summary_df["p_BH_within_gene_set"] < 0.05
)
mi_summary_df["significant_BH_all_33"] = (
    mi_summary_df["p_BH_all_33"] < 0.05
)

# Ordering
mi_summary_df["dataset_label"] = pd.Categorical(
    mi_summary_df["dataset_label"],
    categories=dataset_order,
    ordered=True,
)
mi_summary_df["gene_set"] = pd.Categorical(
    mi_summary_df["gene_set"],
    categories=GENE_SET_ORDER,
    ordered=True,
)

mi_summary_df = (
    mi_summary_df
    .sort_values(["gene_set", "dataset_label"])
    .reset_index(drop=True)
)

summary_path = MI_MATCH_OUT_DIR / "MT_Autophagy_ncMT_expression_matched_MI_summary.csv"
mi_summary_df.to_csv(summary_path, index=False)

display(mi_summary_df)
print("Saved:", summary_path)


## MI-ratio plots

In [ ]:
def plot_mi_ratio(gene_set, q_col="p_BH_all_33"):
    plot_df = (
        mi_summary_df.loc[mi_summary_df["gene_set"] == gene_set]
        .copy()
        .sort_values("dataset_label")
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(plot_df))

    ax.axhline(1, linestyle="--", linewidth=1, color="black")

    sig = plot_df[q_col] < 0.05
    ns = ~sig

    ax.scatter(
        x[ns],
        plot_df.loc[ns, "MI_ratio_observed_over_matched"],
        facecolors="white",
        edgecolors="black",
        s=65,
        zorder=3,
        label="BH q ≥ 0.05",
    )

    ax.scatter(
        x[sig],
        plot_df.loc[sig, "MI_ratio_observed_over_matched"],
        color="black",
        s=65,
        zorder=3,
        label="BH q < 0.05",
    )

    for i, row in plot_df.iterrows():
        q = row[q_col]

        if q < 0.001:
            stars = "***"
        elif q < 0.01:
            stars = "**"
        elif q < 0.05:
            stars = "*"
        else:
            stars = ""

        if stars:
            ax.text(
                i,
                row["MI_ratio_observed_over_matched"] + 0.01,
                stars,
                ha="center",
                va="bottom",
            )

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["dataset_label"], rotation=45, ha="right")
    ax.set_ylabel(f"{gene_set} mean MI / matched mean MI")
    ax.set_xlabel("")
    ax.set_title(f"{gene_set} | significance from {q_col}")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False)

    plt.tight_layout()
    plt.show()


# Primary view: correction across all 33 tests.
# Change q_col to "p_BH_within_gene_set" to reproduce the
# 11-test family used by the old MT-only notebook.
for gene_set in GENE_SET_ORDER:
    plot_mi_ratio(gene_set, q_col="p_BH_all_33")


## Optional: compact result table

In [ ]:
display(
    mi_summary_df[[
        "gene_set",
        "dataset_label",
        "n_target_genes",
        "observed_mean_MI",
        "matched_mean_MI",
        "MI_ratio_observed_over_matched",
        "empirical_p_mean",
        "p_BH_within_gene_set",
        "p_BH_all_33",
    ]]
)
